In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
    EarlyStoppingCallback
)
from datasets import Dataset

TRAIN_FILE = '/content/data/train_augmented_bert.csv'
DEV_FILE = '/content/data/dev.csv'
MODEL_NAME = "google/muril-base-cased"
OUTPUT_DIR = "/content/drive/MyDrive/political-comments/final_aggressive_model"
NUM_LABELS = 7
MAX_LEN = 128
SEED = 42

set_seed(SEED)

label_map = {
    'Substantiated': 0, 'Sarcastic': 1, 'Opinionated': 2,
    'Positive': 3, 'Negative': 4, 'Neutral': 5, 'None of the above': 6
}


class AggressiveWeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to('cuda')

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1_macro = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1_macro}

def load_data(file_path):
    df = pd.read_csv(file_path)
    df = df[df['label'].isin(label_map.keys())]
    df['label'] = df['label'].map(label_map)
    return df

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

train_df = load_data(TRAIN_FILE)
dev_df = load_data(DEV_FILE)


labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)

class_weights[1] = class_weights[1] * 2.0
class_weights[4] = class_weights[4] * 1.5

print(f"Aggressive Weights: {class_weights}")

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
dev_dataset = Dataset.from_pandas(dev_df[['text', 'label']])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_dev = dev_dataset.map(tokenize_function, batched=True)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    report_to="none"
)

trainer = AggressiveWeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting Safe-Aggressive Training...")
trainer.train()


preds_output = trainer.predict(tokenized_dev)
y_preds = np.argmax(preds_output.predictions, axis=1)

id2label = {v: k for k, v in label_map.items()}
print("\nClassification Report (Aggressive + Early Stopping):")
print(classification_report(tokenized_dev['label'], y_preds, target_names=[id2label[i] for i in range(7)]))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✔ Model Saved to: {OUTPUT_DIR}")

In [ ]:
import zipfile
import os
import pandas as pd
from google.colab import drive


drive.mount('/content/drive')


ZIP_PATH = '/content/drive/MyDrive/political-comments/project_data.zip'
EXTRACT_PATH = '/content/data/'

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Files extracted:", os.listdir(EXTRACT_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
from datasets import Dataset


CHECKPOINT_PATH = "/content/drive/MyDrive/political-comments/final_aggressive_model/checkpoint-7657"
DEV_FILE = "/content/data/data/dev.csv"

label_map = {
    'Substantiated': 0, 'Sarcastic': 1, 'Opinionated': 2,
    'Positive': 3, 'Negative': 4, 'Neutral': 5, 'None of the above': 6
}
id2label = {v: k for k, v in label_map.items()}

dev_df = pd.read_csv(DEV_FILE)
dev_df = dev_df[dev_df['label'].isin(label_map.keys())]
dev_df['label'] = dev_df['label'].map(label_map)
test_dataset = Dataset.from_pandas(dev_df)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_PATH)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_test = test_dataset.map(tokenize_function, batched=True)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

print("Predicting labels...")
raw_preds = trainer.predict(tokenized_test)
y_preds = np.argmax(raw_preds.predictions, axis=1)
y_true = tokenized_test['label']

cm = confusion_matrix(y_true, y_preds)
fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[id2label[i] for i in range(len(label_map))]
)

disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix: Political Comment Analysis")
plt.savefig('confusion_matrix.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.show()
